# Prediction Market Overreaction Detector — Data Collection Pipeline

This notebook:
1. Loads and processes Kalshi contract price history from a local CSV
2. Collects and embeds financial news headlines from GDELT for each trading day using an N-day lookback window
3. Assembles a feature tensor combining embeddings, sentiment, and price data

Run cells top-to-bottom. Heavy steps are cached to `./data/` so reruns are fast.

## Section 0: Install Dependencies

In [ ]:
import sys

# Packages without cp314 wheels for their pinned versions are left unpinned
# so uv resolves a release that ships a pre-built wheel for this interpreter.
# ipywidgets is required for tqdm's notebook progress bars.
!uv pip install -q --python "{sys.executable}" \
    pandas==2.2.3 \
    matplotlib \
    gdeltdoc==1.2.0 \
    sentence-transformers \
    transformers \
    torch \
    tqdm==4.67.1 \
    numpy \
    ipywidgets

## Section 1: Config & Setup

In [ ]:
import os
import sys

# ── Configurable parameters ────────────────────────────────────────────────────
N_DAY_WINDOW    = 6                          # lookback window in calendar days
KALSHI_CSV_PATH = "./data/kalshi_contract.csv"
NEWS_KEYWORDS   = [
    "FOMC", "federal reserve", "interest rate",
    "rate cut", "rate hike", "Powell", "Fed",
]
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
SENTIMENT_MODEL = "ProsusAI/finbert"
GDELT_SLEEP_S   = 0.5   # seconds between GDELT requests (rate-limit courtesy)
GDELT_RETRIES   = 3     # number of retry attempts per failed query
GDELT_CACHE     = "./data/gdelt_cache.json"
FEATURES_OUT    = "./data/features.pkl"
# ──────────────────────────────────────────────────────────────────────────────

# Create data directory
os.makedirs("./data", exist_ok=True)
print("Config loaded. ./data/ directory ready.")

## Section 2: Load Kalshi CSV

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ── Guard: check that the CSV exists before going further ─────────────────────
if not os.path.exists(KALSHI_CSV_PATH):
    print(
        "\n⚠️  Kalshi CSV not found!"
        f"\n   Expected: {os.path.abspath(KALSHI_CSV_PATH)}"
        "\n"
        "\n   Please place your Kalshi price-history CSV at that path."
        "\n   The file should have (at minimum) two columns:"
        "\n     • a date/timestamp column  (e.g. 'timestamp' in ISO-8601 format)"
        "\n     • a closing price column   (e.g. 'close' or a contract-name column)"
        "\n"
        "\n   One easy way: export the daily price history from Kalshi's website"
        "\n   and drop the resulting CSV into ./data/kalshi_contract.csv"
    )
    sys.exit(1)

# ── Load raw CSV ──────────────────────────────────────────────────────────────
raw_df = pd.read_csv(KALSHI_CSV_PATH)
print("Raw columns:", list(raw_df.columns))
print("\nFirst few rows:")
display(raw_df.head())

In [ ]:
def normalize_kalshi_df(raw: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize whatever column layout Kalshi exports into a clean DataFrame
    with columns: [date, close_price, daily_change].

    Strategy:
      - The first column is assumed to be the timestamp.
      - The second column is assumed to be the price series
        (works whether Kalshi names it 'close', a contract title, etc.).
    """
    df = raw.copy()

    date_col  = df.columns[0]
    price_col = df.columns[1]

    df["date"]        = pd.to_datetime(df[date_col], utc=True).dt.normalize().dt.tz_localize(None)
    df["close_price"] = pd.to_numeric(df[price_col], errors="coerce")

    df = (
        df[["date", "close_price"]]
        .dropna(subset=["close_price"])
        .sort_values("date")
        .reset_index(drop=True)
    )

    # Day-over-day price difference; first row gets NaN → fill 0
    df["daily_change"] = df["close_price"].diff().fillna(0)

    return df


kalshi_df = normalize_kalshi_df(raw_df)

# Derive START_DATE / END_DATE from the data itself
START_DATE = kalshi_df["date"].min()
END_DATE   = kalshi_df["date"].max()

print("\nClean Kalshi DataFrame:")
display(kalshi_df.head())
print(f"\nDate range : {START_DATE.date()} → {END_DATE.date()}")
print(f"Trading days: {len(kalshi_df)}")
print(f"Close price — min: {kalshi_df.close_price.min():.2f}  "
      f"max: {kalshi_df.close_price.max():.2f}  "
      f"mean: {kalshi_df.close_price.mean():.2f}")

# Largest single-day moves
top_moves = kalshi_df.reindex(
    kalshi_df["daily_change"].abs().nlargest(5).index
)[["date", "close_price", "daily_change"]]
print("\nLargest single-day moves:")
display(top_moves)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(kalshi_df["date"], kalshi_df["close_price"], linewidth=1.5, color="steelblue")
ax.set_title("Kalshi Contract — Full Price History", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Price (¢)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Section 3: GDELT News Collection

In [ ]:
import json
import time
from datetime import timedelta
from gdeltdoc import GdeltDoc, Filters

gd = GdeltDoc()


def build_gdelt_filters(window_start: pd.Timestamp, window_end: pd.Timestamp) -> Filters:
    """Construct a GDELT Filters object for the given date window."""
    return Filters(
        keyword     = " OR ".join(NEWS_KEYWORDS),
        start_date  = window_start.strftime("%Y-%m-%d"),
        end_date    = window_end.strftime("%Y-%m-%d"),
        language    = "English",
        num_records = 250,   # GDELT hard cap per query
    )


def fetch_gdelt_titles(window_start: pd.Timestamp, window_end: pd.Timestamp) -> list[str]:
    """
    Query GDELT for article titles in [window_start, window_end].
    Returns a deduplicated list of English headline strings.
    Retries up to GDELT_RETRIES times on failure.
    """
    for attempt in range(1, GDELT_RETRIES + 1):
        try:
            f = build_gdelt_filters(window_start, window_end)
            articles = gd.article_search(f)
            if articles is None or articles.empty:
                return []
            # Deduplicate by exact title (case-folded)
            seen, titles = set(), []
            for t in articles["title"].dropna():
                key = t.strip().lower()
                if key not in seen:
                    seen.add(key)
                    titles.append(t.strip())
            return titles
        except Exception as exc:
            print(f"  [GDELT] attempt {attempt}/{GDELT_RETRIES} failed: {exc}")
            if attempt < GDELT_RETRIES:
                time.sleep(2 ** attempt)  # exponential back-off
    return []

In [ ]:
from tqdm.auto import tqdm  # auto picks notebook widgets when available, text bar otherwise

# ── Load cache if it exists (avoids re-querying GDELT on reruns) ──────────────
if os.path.exists(GDELT_CACHE):
    with open(GDELT_CACHE, "r") as fh:
        # Keys are ISO date strings; convert back to pd.Timestamp after load
        raw_cache = json.load(fh)
    gdelt_results = {pd.Timestamp(k): v for k, v in raw_cache.items()}
    print(f"Loaded GDELT cache from {GDELT_CACHE} ({len(gdelt_results)} entries).")
else:
    gdelt_results = {}
    print("No cache found — will query GDELT from scratch.")

# ── Query for any trading days not yet in the cache ───────────────────────────
trading_days = kalshi_df["date"].tolist()
missing_days = [d for d in trading_days if d not in gdelt_results]
print(f"{len(missing_days)} trading day(s) need GDELT queries.")

for trade_date in tqdm(missing_days, desc="Querying GDELT"):
    window_end   = trade_date
    window_start = trade_date - timedelta(days=N_DAY_WINDOW - 1)
    titles = fetch_gdelt_titles(window_start, window_end)
    gdelt_results[trade_date] = titles
    time.sleep(GDELT_SLEEP_S)

# ── Persist updated cache ─────────────────────────────────────────────────────
with open(GDELT_CACHE, "w") as fh:
    # Serialize Timestamps as ISO strings
    json.dump({str(k.date()): v for k, v in gdelt_results.items()}, fh, indent=2)
print(f"Cache saved to {GDELT_CACHE}.")

In [ ]:
# ── Summary statistics for GDELT collection ───────────────────────────────────
article_counts = {d: len(gdelt_results.get(d, [])) for d in trading_days}
total_unique   = sum(article_counts.values())
avg_per_window = total_unique / max(len(trading_days), 1)
zero_days      = [d for d, cnt in article_counts.items() if cnt == 0]

print(f"Total article-window pairs collected : {total_unique}")
print(f"Average articles per window          : {avg_per_window:.1f}")
print(f"Trading days with zero articles      : {len(zero_days)}")
if zero_days:
    print("  ⚠️  Zero-article dates (will use zero vectors):")
    for d in zero_days:
        print(f"     {d.date()}")

## Section 4: Headline Embedding & Sentiment

In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

# ── Load models ───────────────────────────────────────────────────────────────
print(f"Loading embedding model: {EMBEDDING_MODEL}")
embedder = SentenceTransformer(EMBEDDING_MODEL)
EMBEDDING_DIM = embedder.get_sentence_embedding_dimension()
print(f"  Embedding dimension: {EMBEDDING_DIM}")

print(f"\nLoading sentiment model: {SENTIMENT_MODEL}")
sentiment_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL)
sentiment_model     = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL)
sentiment_model.eval()
print("  FinBERT loaded. Labels: positive / negative / neutral")

In [ ]:
# ── Embedding helpers ─────────────────────────────────────────────────────────

def embed_headlines(headlines: list[str]) -> np.ndarray:
    """
    Compute the mean sentence embedding for a list of headlines.
    Returns a 1-D array of shape (EMBEDDING_DIM,).
    If the list is empty, returns a zero vector.
    """
    if not headlines:
        return np.zeros(EMBEDDING_DIM, dtype=np.float32)
    vecs = embedder.encode(headlines, show_progress_bar=False, convert_to_numpy=True)
    return vecs.mean(axis=0).astype(np.float32)


def sentiment_scores(headlines: list[str], batch_size: int = 16) -> np.ndarray:
    """
    Run FinBERT on each headline and average the softmax probabilities.
    Returns a 1-D array [p_positive, p_negative, p_neutral].
    If the list is empty, returns uniform (1/3, 1/3, 1/3).

    FinBERT label order: 0=positive, 1=negative, 2=neutral
    """
    UNIFORM = np.array([1/3, 1/3, 1/3], dtype=np.float32)
    if not headlines:
        return UNIFORM

    all_probs = []
    for i in range(0, len(headlines), batch_size):
        batch = headlines[i : i + batch_size]
        enc = sentiment_tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=128,
            padding=True,
        )
        with torch.no_grad():
            logits = sentiment_model(**enc).logits
        probs = F.softmax(logits, dim=-1).numpy()  # shape (batch, 3)
        all_probs.append(probs)

    return np.vstack(all_probs).mean(axis=0).astype(np.float32)  # (3,)


def build_day_features(
    day: pd.Timestamp,
    headlines: list[str],
    price_change: float,
) -> np.ndarray:
    """
    Combine embedding + sentiment + price_change into a single feature vector.
    Shape: (EMBEDDING_DIM + 3 + 1,)
    """
    emb  = embed_headlines(headlines)              # (EMBEDDING_DIM,)
    sent = sentiment_scores(headlines)             # (3,)
    pc   = np.array([price_change], dtype=np.float32)  # (1,)
    return np.concatenate([emb, sent, pc])         # (D,)

In [ ]:
PER_DAY_CACHE_PATH = "./data/gdelt_per_day_cache.json"

if os.path.exists(PER_DAY_CACHE_PATH):
    with open(PER_DAY_CACHE_PATH) as fh:
        per_day_headlines: dict[str, list[str]] = json.load(fh)
    print(f"Loaded per-day cache ({len(per_day_headlines)} entries).")
else:
    per_day_headlines = {}

# Identify all individual calendar days we need
needed_days: set[pd.Timestamp] = set()
for trade_date in trading_days:
    for offset in range(N_DAY_WINDOW):
        needed_days.add(trade_date - timedelta(days=offset))

missing_individual = [
    d for d in sorted(needed_days)
    if str(d.date()) not in per_day_headlines
]
print(f"{len(missing_individual)} individual day(s) need separate GDELT single-day queries.")

for day in tqdm(missing_individual, desc="Per-day GDELT queries"):
    titles = fetch_gdelt_titles(day, day)
    per_day_headlines[str(day.date())] = titles
    time.sleep(GDELT_SLEEP_S)

with open(PER_DAY_CACHE_PATH, "w") as fh:
    json.dump(per_day_headlines, fh, indent=2)
print("Per-day cache saved.")

In [ ]:
# ── Build feature tensors for every trading day ───────────────────────────────
FEATURE_DIM = EMBEDDING_DIM + 3 + 1  # embedding + sentiment + price_change
dataset: list[dict] = []
zero_vector_days: list[tuple[pd.Timestamp, pd.Timestamp]] = []  # (trade_date, window_day)

for _, row in tqdm(kalshi_df.iterrows(), total=len(kalshi_df), desc="Building features"):
    trade_date   = row["date"]
    close_price  = row["close_price"]
    daily_change = row["daily_change"]

    # Collect one feature vector per day in the lookback window
    # Window: [trade_date - (N-1) days, ..., trade_date]
    window_features: list[np.ndarray] = []

    for offset in reversed(range(N_DAY_WINDOW)):  # oldest → newest
        window_day  = trade_date - timedelta(days=offset)
        day_key     = str(window_day.date())
        headlines   = per_day_headlines.get(day_key, [])

        # Look up that window-day's price change from kalshi_df (if available)
        price_row = kalshi_df[kalshi_df["date"] == window_day]
        w_price_change = float(price_row["daily_change"].iloc[0]) if not price_row.empty else 0.0

        if not headlines:
            zero_vector_days.append((trade_date, window_day))

        feat = build_day_features(window_day, headlines, w_price_change)
        window_features.append(feat)

    feature_tensor = np.stack(window_features, axis=0)  # (N_DAY_WINDOW, FEATURE_DIM)
    dataset.append({
        "date"         : trade_date,
        "features"     : feature_tensor,
        "price"        : close_price,
        "daily_change" : daily_change,
    })

print(f"\nBuilt {len(dataset)} samples. Feature tensor shape per sample: "
      f"({N_DAY_WINDOW}, {FEATURE_DIM})")
print(f"Zero-vector days (no headlines): {len(zero_vector_days)}")

In [ ]:
import pickle

with open(FEATURES_OUT, "wb") as fh:
    pickle.dump(dataset, fh)

print(f"Dataset saved to {FEATURES_OUT}")
print(f"  Samples        : {len(dataset)}")
print(f"  Feature shape  : {dataset[0]['features'].shape}  (N_DAY_WINDOW × FEATURE_DIM)")

## Section 5: Data Validation & Summary

In [ ]:
import random

# ── Overall shape report ──────────────────────────────────────────────────────
n_samples, (n_win, n_feat) = len(dataset), dataset[0]["features"].shape
print("=" * 60)
print(f"Final dataset summary")
print("=" * 60)
print(f"  Number of samples  : {n_samples}")
print(f"  Window size        : {n_win} days")
print(f"  Feature dim / day  : {n_feat}  "
      f"({EMBEDDING_DIM} embed + 3 sentiment + 1 price_Δ)")
print(f"  Full tensor shape  : ({n_samples}, {n_win}, {n_feat})")

In [ ]:
# ── Spot-check 3 random samples ───────────────────────────────────────────────
spot_indices = random.sample(range(len(dataset)), min(3, len(dataset)))

for idx in spot_indices:
    sample = dataset[idx]
    trade_date = sample["date"]
    print("\n" + "─" * 60)
    print(f"Sample idx={idx}  |  Trade date: {trade_date.date()}")
    print(f"  Close price  : {sample['price']:.2f}¢")
    print(f"  Daily change : {sample['daily_change']:+.2f}¢")

    # Show headlines and sentiment for each day in the window
    for offset in range(N_DAY_WINDOW):
        window_day = trade_date - timedelta(days=N_DAY_WINDOW - 1 - offset)
        day_key    = str(window_day.date())
        headlines  = per_day_headlines.get(day_key, [])
        feat_row   = sample["features"][offset]

        # Last 3 elements: sentiment triplet; element before that: price_change
        price_feat = feat_row[-4]
        sent_feat  = feat_row[-3:]  # [pos, neg, neu]

        print(f"\n  Window day [{offset}]: {window_day.date()} "
              f"({len(headlines)} headlines)")
        print(f"    Sentiment  pos={sent_feat[0]:.3f}  "
              f"neg={sent_feat[1]:.3f}  neu={sent_feat[2]:.3f}")
        for h in headlines[:3]:  # show at most 3 example headlines
            print(f"    • {h[:100]}")
        if len(headlines) > 3:
            print(f"    … and {len(headlines) - 3} more")

In [ ]:
# ── Histogram: article counts per window ──────────────────────────────────────
window_counts = [len(gdelt_results.get(s["date"], [])) for s in dataset]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(window_counts, bins=20, color="steelblue", edgecolor="white", linewidth=0.5)
ax.set_title("Distribution of Article Counts per N-day Window", fontsize=13)
ax.set_xlabel("Number of articles in window")
ax.set_ylabel("Number of trading days")
ax.axvline(x=0.5, color="red", linestyle="--", linewidth=1, label="Zero-article threshold")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nWindow coverage stats:")
print(f"  Min articles/window  : {min(window_counts)}")
print(f"  Max articles/window  : {max(window_counts)}")
print(f"  Mean articles/window : {sum(window_counts)/len(window_counts):.1f}")
print(f"  Zero-article windows : {sum(1 for c in window_counts if c == 0)}")